In [ ]:
"""
End-to-End LiDAR Transformation & Visualization Toolkit
Ingests ROS 2 bags, transforms PointCloud2 data to base_footprint,
and provides options to stream, visualize, or save the data as .npy files.
"""
import os
import time
from pathlib import Path
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

from rosbags.highlevel import AnyReader
from rosbags.typesys import Stores, get_typestore

# ==========================================
# 1. CORE MATH (Replicating ROS TF2)
# ==========================================
def _quat_to_matrix(qx, qy, qz, qw):
    r = np.eye(3)
    r[0, 0] = 1.0 - 2.0*(qy**2 + qz**2)
    r[0, 1] = 2.0*(qx*qy - qz*qw)
    r[0, 2] = 2.0*(qx*qz + qy*qw)
    r[1, 0] = 2.0*(qx*qy + qz*qw)
    r[1, 1] = 1.0 - 2.0*(qx**2 + qz**2)
    r[1, 2] = 2.0*(qy*qz - qx*qw)
    r[2, 0] = 2.0*(qx*qz - qy*qw)
    r[2, 1] = 2.0*(qy*qz + qx*qw)
    r[2, 2] = 1.0 - 2.0*(qx**2 + qy**2)
    return r

def _build_transform(x, y, z, qx, qy, qz, qw):
    mat = np.eye(4)
    mat[:3, :3] = _quat_to_matrix(qx, qy, qz, qw)
    mat[0, 3] = x
    mat[1, 3] = y
    mat[2, 3] = z
    return mat

def get_robot_transform_matrix():
    # TF Tree: base_footprint -> base_link -> flat_rotated -> rotated -> hesai_lidar
    t_footprint_link = _build_transform(0.0, 0.0, 0.254,  0.0, 0.0, 0.0, 1.0)
    t_link_flat      = _build_transform(1.36, -0.01, 0.294,  0.0, 0.0, 0.0, 1.0)
    t_flat_rot       = _build_transform(0.0, 0.0, 0.0,  0.0, 0.1348509, 0.0, 0.9908659)
    t_rot_lidar      = _build_transform(0.0, 0.0, 0.0,  0.0, 0.0, -1.0, 0.0)

    return t_footprint_link @ t_link_flat @ t_flat_rot @ t_rot_lidar

def transform_point_cloud(raw_points):
    matrix = get_robot_transform_matrix()
    ones = np.ones((raw_points.shape[0], 1))
    points_homogeneous = np.hstack([raw_points, ones])
    transformed_homogeneous = (matrix @ points_homogeneous.T).T
    return transformed_homogeneous[:, :3]

# ==========================================
# 2. ROS BAG INGESTION
# ==========================================
def _extract_xyz_from_msg(msg):
    x_off = next(f.offset for f in msg.fields if f.name == 'x')
    y_off = next(f.offset for f in msg.fields if f.name == 'y')
    z_off = next(f.offset for f in msg.fields if f.name == 'z')

    dt = np.dtype({
        'names': ['x', 'y', 'z'],
        'formats': ['<f4', '<f4', '<f4'],
        'offsets': [x_off, y_off, z_off],
        'itemsize': msg.point_step
    })

    parsed = np.frombuffer(msg.data, dtype=dt)
    xyz = np.column_stack((parsed['x'], parsed['y'], parsed['z']))
    mask = np.isfinite(xyz).all(axis=1)
    return xyz[mask]

def ingest_rosbag(bag_path, topic_name):
    """Generates transformed ML-ready NumPy arrays (N x 3) one scan at a time."""
    typestore = get_typestore(Stores.ROS2_HUMBLE)
    bag_path_obj = Path(bag_path)

    if not bag_path_obj.exists():
        raise FileNotFoundError(f"Cannot find ROS bag at: {bag_path}")

    with AnyReader([bag_path_obj], default_typestore=typestore) as reader:
        for connection, timestamp, rawdata in reader.messages():
            if connection.topic == topic_name:
                msg = reader.deserialize(rawdata, connection.msgtype)
                raw_points = _extract_xyz_from_msg(msg)
                if len(raw_points) == 0:
                    continue
                yield transform_point_cloud(raw_points)

# ==========================================
# 3. EXPORTING DATA
# ==========================================
def save_dataset_to_disk(bag_path, topic_name, output_dir="./transformed_npy_dataset"):
    """Extracts the entire bag and saves each transformed scan as a .npy file."""
    os.makedirs(output_dir, exist_ok=True)
    print(f"Extracting and saving dataset to '{output_dir}'...")

    stream = ingest_rosbag(bag_path, topic_name)
    count = 0

    for scan in stream:
        filename = os.path.join(output_dir, f"scan_{count:05d}.npy")
        np.save(filename, scan)
        count += 1
        if count % 100 == 0:
            print(f"Saved {count} scans to disk...")

    print(f"Complete! Successfully saved {count} .npy files.")
    return output_dir

# ==========================================
# 4. VISUALIZATION
# ==========================================
def _colorize_by_z(points):
    z_norm = np.clip((points[:, 2] - (-2.0)) / (5.0 - (-2.0)), 0.0, 1.0)
    return plt.get_cmap('jet')(z_norm)[:, :3]

def _create_grid():
    grid_lines, grid_points = [], []
    for i in range(-50, 55, 5):
        grid_points.extend([[-50, i, 0], [50, i, 0], [i, -50, 0], [i, 50, 0]])
        idx = len(grid_points)
        grid_lines.extend([[idx-4, idx-3], [idx-2, idx-1]])
    grid = o3d.geometry.LineSet()
    grid.points = o3d.utility.Vector3dVector(grid_points)
    grid.lines = o3d.utility.Vector2iVector(grid_lines)
    grid.colors = o3d.utility.Vector3dVector([[0.3, 0.3, 0.35]] * len(grid_lines))
    return grid

def play_rosbag_stream(scan_generator):
    """Plays back a stream of transformed point clouds like a video."""
    print("Opening 3D Viewer... Press 'Q' to close.")

    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="ML Teammate ROS Bag Viewer", width=1280, height=720)
    pcd = o3d.geometry.PointCloud()

    try:
        first_scan = next(scan_generator)
    except StopIteration:
        print("No messages found on that topic!")
        return

    pcd.points = o3d.utility.Vector3dVector(first_scan)
    pcd.colors = o3d.utility.Vector3dVector(_colorize_by_z(first_scan))

    vis.add_geometry(pcd)
    vis.add_geometry(_create_grid())
    vis.add_geometry(o3d.geometry.TriangleMesh.create_coordinate_frame(size=3.0, origin=[0,0,0]))

    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.08])
    opt.point_size = 2.0

    vis.poll_events()
    vis.update_renderer()
    ctr = vis.get_view_control()
    ctr.set_lookat([0.0, 0.0, 0.0])
    ctr.set_up([0.0, 0.0, 1.0])
    ctr.set_front([-1.0, -0.5, 0.8])
    ctr.set_zoom(0.1)

    for points in scan_generator:
        pcd.points = o3d.utility.Vector3dVector(points)
        pcd.colors = o3d.utility.Vector3dVector(_colorize_by_z(points))
        vis.update_geometry(pcd)
        vis.poll_events()
        vis.update_renderer()
        time.sleep(0.05)

    vis.destroy_window()

# ==========================================
# 5. HOW TO USE IT
# ==========================================
if __name__ == "__main__":







    # Update these paths before running!
    MY_BAG_PATH = r"C:/Users/joven\Downloads/Lidar ROS Bags (1)/Lidar ROS Bags/rosbag2_2026_02_19-10_15_38_D"
    MY_TOPIC = '/lidar_points'

    # --- OPTION A: Save everything to disk as .npy files ---
    # Uncomment the line below to permanently extract the dataset to a folder

    OUTPUT_FOLDER = r"C:\Users\joven\OneDrive\Documents\Ml-Projects\adaptive-dbscan-core\Lidar_Data\Lidar_data_testD"

    #save_dataset_to_disk(MY_BAG_PATH, MY_TOPIC, output_dir=OUTPUT_FOLDER)



    # --- OPTION B: Stream and visualize instantly ---
    # Watch the point clouds play back directly from the bag
    
    my_stream = ingest_rosbag(MY_BAG_PATH, MY_TOPIC)
    play_rosbag_stream(my_stream)


# USING IT FOR ML:
"""
import torch
from lidar_transform_utils import ingest_rosbag

# Setup the stream from the ROS bag
bag_path = "path/to/rosbag"
topic = "/fsaivehicle/xt32/point_cloud"
lidar_stream = ingest_rosbag(bag_path, topic)

# Iterate through the bag directly in the ML pipeline
for numpy_scan in lidar_stream:
    q
    # Convert numpy array to PyTorch tensor
    tensor_scan = torch.tensor(numpy_scan, dtype=torch.float32)
    
    # Feed to Neural Network
    # predictions = my_ml_model(tensor_scan)
    # loss.backward()
    # ... etc

"""

Opening 3D Viewer... Press 'Q' to close.


'\nimport torch\nfrom lidar_transform_utils import ingest_rosbag\n\n# Setup the stream from the ROS bag\nbag_path = "path/to/rosbag"\ntopic = "/fsaivehicle/xt32/point_cloud"\nlidar_stream = ingest_rosbag(bag_path, topic)\n\n# Iterate through the bag directly in the ML pipeline\nfor numpy_scan in lidar_stream:\n    q\n    # Convert numpy array to PyTorch tensor\n    tensor_scan = torch.tensor(numpy_scan, dtype=torch.float32)\n    \n    # Feed to Neural Network\n    # predictions = my_ml_model(tensor_scan)\n    # loss.backward()\n    # ... etc\n\n'

: 